# Deep Q-Learning on CartPole

**Author:** Muhammad Umair Bashir  
**Project Type:** Reinforcement Learning Portfolio Project

This notebook presents a clean and portfolio-ready **Deep Q-Learning (DQN)** project using the **CartPole-v1** environment from Gymnasium and the **stable-baselines3** library.

The project is designed to demonstrate:
- reinforcement learning fundamentals,
- training a Deep Q-Network agent,
- evaluating agent performance,
- visualizing rewards and learning behaviour,
- presenting results in a professional portfolio format.


## Project Objectives

- Train a Deep Q-Network agent on CartPole
- Evaluate policy performance over multiple episodes
- Track and visualize training behaviour
- Compare training and evaluation statistics
- Build a polished reinforcement learning project for GitHub and LinkedIn


## Why CartPole?

CartPole is one of the most popular reinforcement learning benchmark environments.  
The agent must keep a pole balanced on a moving cart by choosing left or right actions.

This environment is useful because it helps demonstrate:
- state-based decision making,
- reward-driven learning,
- policy improvement over time,
- deep reinforcement learning in a simple but meaningful setting.


In [ ]:
# If running in Google Colab, keep this installation cell
!pip -q install stable-baselines3 gymnasium pandas matplotlib

import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

SEED = 42
np.random.seed(SEED)


## Environment Setup

In [ ]:
env = Monitor(gym.make("CartPole-v1"))
eval_env = gym.make("CartPole-v1")

obs, info = env.reset(seed=SEED)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Initial observation:", obs)


## DQN Configuration

The model below uses an MLP policy and standard DQN components such as:
- replay buffer,
- target network updates,
- epsilon-greedy exploration,
- discounted future rewards.


In [ ]:
class RewardLoggerCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        for info in infos:
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_lengths.append(info["episode"]["l"])
        return True

reward_callback = RewardLoggerCallback()

model = DQN(
    "MlpPolicy",
    env,
    learning_rate=1e-3,
    buffer_size=50000,
    learning_starts=1000,
    batch_size=64,
    gamma=0.99,
    train_freq=4,
    target_update_interval=1000,
    exploration_fraction=0.20,
    exploration_final_eps=0.05,
    verbose=0,
    seed=SEED
)


## Model Training

In [ ]:
total_timesteps = 40000
model.learn(total_timesteps=total_timesteps, callback=reward_callback)
print("DQN training completed.")

## Training Reward Summary

In [ ]:
reward_df = pd.DataFrame({
    "episode": np.arange(1, len(reward_callback.episode_rewards) + 1),
    "reward": reward_callback.episode_rewards,
    "episode_length": reward_callback.episode_lengths
})
reward_df["rolling_reward_20"] = reward_df["reward"].rolling(20).mean()
reward_df.head()

### Episode Reward Curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(reward_df["episode"], reward_df["reward"], alpha=0.5, label="Episode reward")
plt.plot(reward_df["episode"], reward_df["rolling_reward_20"], linewidth=2, label="Rolling mean (20)")
plt.title("CartPole Training Rewards")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Episode Length Curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(reward_df["episode"], reward_df["episode_length"], alpha=0.7)
plt.title("CartPole Episode Length During Training")
plt.xlabel("Episode")
plt.ylabel("Episode Length")
plt.grid(True, alpha=0.3)
plt.show()

### Reward Distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(reward_df["reward"], bins=20)
plt.title("Distribution of Training Episode Rewards")
plt.xlabel("Reward")
plt.ylabel("Count")
plt.grid(True, alpha=0.3)
plt.show()

## Evaluation Function

In [ ]:
def eval_model(model, env, episodes=30):
    scores = []
    for _ in range(episodes):
        obs, _ = env.reset()
        done = False
        total = 0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total += reward
        scores.append(total)
    return float(np.mean(scores)), float(np.std(scores)), scores

mean_score, std_score, eval_scores = eval_model(model, eval_env, episodes=30)

print("Evaluation mean score:", mean_score)
print("Evaluation standard deviation:", std_score)

## Evaluation Results

In [ ]:
eval_df = pd.DataFrame({
    "episode": np.arange(1, len(eval_scores) + 1),
    "score": eval_scores
})
eval_df

### Evaluation Score by Episode

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(eval_df["episode"], eval_df["score"], marker="o")
plt.axhline(eval_df["score"].mean(), linestyle="--", label=f"Mean = {eval_df['score'].mean():.2f}")
plt.title("CartPole Evaluation Scores")
plt.xlabel("Evaluation Episode")
plt.ylabel("Score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Training vs Evaluation Comparison

In [ ]:
comparison_df = pd.DataFrame({
    "Metric": ["Mean reward", "Std deviation", "Best score"],
    "Training": [
        reward_df["reward"].mean(),
        reward_df["reward"].std(),
        reward_df["reward"].max()
    ],
    "Evaluation": [
        eval_df["score"].mean(),
        eval_df["score"].std(),
        eval_df["score"].max()
    ]
})
comparison_df

In [ ]:
plot_df = comparison_df.set_index("Metric")
plot_df.plot(kind="bar", figsize=(9, 4))
plt.title("Training vs Evaluation Metrics")
plt.ylabel("Value")
plt.xticks(rotation=0)
plt.grid(True, axis="y", alpha=0.3)
plt.legend()
plt.show()

## Hyperparameter Summary

In [ ]:
hyperparams = pd.DataFrame({
    "Parameter": [
        "learning_rate", "buffer_size", "learning_starts", "batch_size",
        "gamma", "train_freq", "target_update_interval",
        "exploration_fraction", "exploration_final_eps", "total_timesteps"
    ],
    "Value": [
        1e-3, 50000, 1000, 64,
        0.99, 4, 1000,
        0.20, 0.05, total_timesteps
    ]
})
hyperparams

## Key Insights

- The DQN agent improves over time as episode rewards rise during training.
- The rolling reward curve helps reveal learning progress more clearly than raw episode rewards alone.
- Evaluation across multiple episodes gives a more stable picture of final policy quality.
- CartPole is a strong introductory benchmark for understanding value-based deep reinforcement learning.
- Comparing training and evaluation metrics helps assess both learning behaviour and general policy performance.


## Conclusion

This project demonstrates a complete deep reinforcement learning workflow using a **Deep Q-Network** on the **CartPole** environment.

From a portfolio perspective, this project shows:
- understanding of reinforcement learning concepts,
- practical use of stable-baselines3,
- training and evaluation of a DQN agent,
- professional visualization of results,
- clear communication of insights in notebook form.


## Next Possible Improvements

- Compare DQN with PPO or A2C
- Run multiple seeds and compare stability
- Save and reload the trained policy
- Add video rendering of the trained agent
- Track more training diagnostics over time
